# Trabajo Práctico MBID08 Visualización de Datos

Alumno: Luis Jesus Trujillo La Torre

VIU: luistrujillolatorre10



# DESCRIPCIÓN

Este proyecto es una solución integral de Data Engineering y Business Intelligence diseñada para analizar el mercado global del cine. El flujo de trabajo abarca desde la limpieza programática en Python hasta la publicación de resultados en la web, integrando herramientas líderes en la industria para transformar un dataset crudo (MyMovieDB) en una plataforma de consulta interactiva.

#FUENTES

URL de Kaggle: https://www.kaggle.com/datasets/disham993/9000-movies-dataset

URL de Stackblitz: https://stackblitz.com/~/github.com/ltrujillolatorre/viu

URL de Google Colab: https://colab.research.google.com/drive/1UrhLJZcCVR1goVqSdoMAf_vUiU8cat1e?usp=sharing

# CARACTERÍSTICAS DEL DATASET

Características del conjunto de datos:

>Release_Date: Fecha de estreno de la película.

>Year: Año de estreno de la película.

>Title: Nombre de la película.

>Overview: Breve resumen de la película.

>Popularity: Es una métrica muy importante calculada por los desarrolladores de TMDB en función del número de visualizaciones por día, votos por día, número de usuarios que la marcaron como "favorita" y "lista de seguimiento" para los datos, fecha de lanzamiento y otras métricas más.

>Vote_Count: Total de votos recibidos de los espectadores.

>Vote_Average: Calificación promedio basada en el número de votos y el número de espectadores sobre 10.

>Original_Language: Idioma original de las películas. La versión doblada no se considera en idioma original.

>Genre: Categorías en las que se puede clasificar la película.

>Poster_Url: URL del póster de la película.

Expresiones de gratitud
Un agradecimiento especial al Sr. Nitish Singh de CampusX ( https://www.youtube.com/channel/UCCWi3hpnq_Pe03nGxuS7isg ) por crear tutoriales increíbles y fáciles de seguir.

Inspiración
Se puede crear un sistema de recomendación utilizando los datos CSV.

Fuente:
Los datos CSV se obtuvieron mediante la API https://developers.themoviedb.org/3/movies/get-popular-movies y se limpiaron utilizando las bibliotecas Pandas y Numpy en Python.

1. Preparación del entorno y la ingesta de datos

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [ ]:
import pandas as pd
from google.colab import files # Necesario para descargar en Colab

# 1. CARGA DE DATOS
df = pd.read_csv('mymoviedb.csv', engine='python')

# --- NUEVA OPERACIÓN: ELIMINAR COLUMNA NO DESEADA ---
# Eliminamos 'Poster_Url' para optimizar el dataset
if 'Poster_Url' in df.columns:
    df = df.drop(columns=['Poster_Url'])
# ---------------------------------------------------

# 2. LIMPIEZA INICIAL Y TRANSFORMACIÓN
cols_numericas = ['Vote_Average', 'Popularity', 'Vote_Count']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')

# --- AGREGAR Y POSICIONAR COLUMNA 'Year' ---
df['Year'] = df['Release_Date'].dt.year

columnas = list(df.columns)
if 'Release_Date' in columnas and 'Year' in columnas:
    idx = columnas.index('Release_Date')
    columnas.insert(idx + 1, columnas.pop(columnas.index('Year')))
    df = df[columnas]
# ------------------------------------------

# Eliminar filas con datos críticos nulos
df = df.dropna(subset=['Title', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language' , 'Genre'])

# 3. EXPORTACIÓN Y DESCARGA
nombre_archivo = 'mymoviedb_limpio.csv'

# Guardamos el archivo en el sistema de archivos temporal de Colab
df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')

# Iniciamos la descarga en tu ordenador
files.download(nombre_archivo)

print(f"El archivo {nombre_archivo} se ha procesado exitosamente.")
print(f"Cambios realizados: Columna 'Poster_Url' eliminada y columna 'Year' reubicada.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

El archivo mymoviedb_limpio.csv se ha procesado exitosamente.
Cambios realizados: Columna 'Poster_Url' eliminada y columna 'Year' reubicada.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Carga de datos
df = pd.read_csv('mymoviedb_limpio.csv', engine='python')

# Aseguramos tipos de datos
df['Popularity'] = pd.to_numeric(df['Popularity'], errors='coerce')
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')

# 2. Selección del Top 10
top_movies = df.sort_values(by='Popularity', ascending=False).head(10)

# 3. Creación del gráfico
fig = px.bar(
    top_movies,
    x='Popularity',
    y='Title',
    color='Popularity',
    orientation='h',
    title='Top 10 Películas más Populares y su Fecha de Estreno',
    hover_data={'Release_Date': True, 'Popularity': ':.2f'},
    labels={'Popularity': 'Índice de Popularidad', 'Title': 'Título de la Película'},
    color_continuous_scale='Viridis'
)

# 4. Ajustes estéticos para FONDO BLANCO
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    template='plotly_white',           # CAMBIO: Plantilla blanca
    font=dict(color="#2c3e50"),       # CAMBIO: Color de fuente elegante (azul oscuro/gris)
    title_font=dict(size=22, color="black"), # Título más destacado
    height=600,
    margin=dict(l=150),
    plot_bgcolor='white',              # Asegura fondo blanco en el área de trazado
    paper_bgcolor='white'              # Asegura fondo blanco en el borde del gráfico
)

# Opcional: Agregar bordes a las barras para que resalten más
fig.update_traces(marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.9)

fig.show()

# ==========================================
# 3. EXPORTACIÓN Y DESCARGA
# ==========================================

# A. Guardar CSV Limpio
nombre_csv = 'mymoviedb_limpio.csv'
df.to_csv(nombre_csv, index=False, encoding='utf-8-sig')

# B. Guardar HTML Interactivo
nombre_html = 'reporte_popularidad.html'
fig.write_html(nombre_html)

# C. Descargar archivos automáticamente
print("\n--- Iniciando descarga de archivos ---")
files.download(nombre_csv)
files.download(nombre_html)

# D. Código para STACKBLITZ
# Generamos el div con el CDN de Plotly incluido para que funcione al pegar
codigo_stackblitz = fig.to_html(full_html=False, include_plotlyjs='cdn')

print("\n" + "="*50)
print("COPIA EL SIGUIENTE CÓDIGO PARA TU STACKBLITZ")
print("="*50 + "\n")
print(codigo_stackblitz)


--- Iniciando descarga de archivos ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPIA EL SIGUIENTE CÓDIGO PARA TU STACKBLITZ

<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>                <div id="d5f10eaa-8749-42f9-83c3-56b01eec64c0" class="plotly-graph-div" style="height:600px; width:100%;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("d5f10eaa-8749-42f9-83c3-56b01eec64c0")) {                    Plotly.newPlot(                        "d5f10eaa-8749-42f9-83c3-56b01eec64c0",                        [{"alignmentgroup":"True","customdata":[["2021-12-15T00:00:00"],["2022-03-01T00:00:00"],["2022-02-25T00:00:00"],["2021-11-24T00:00:00"],["2021-12-22T00:00:00"],["2022-01-07T00:00:00"],["2022-01-12T00:00:00"],["2022-02-10T00:00:00"],["2022-02-17T00:00:00"],["2021-11-03T00:0

In [ ]:
import pandas as pd
import plotly.express as px
from google.colab import files

# 1. CARGA Y LIMPIEZA MULTIDIMENSIONAL
try:
    df = pd.read_csv('mymoviedb.csv', engine='python')
except FileNotFoundError:
    print("❌ Error: No se encontró 'mymoviedb.csv'.")
    raise

# Conversiones de tipos
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
df['Popularity'] = pd.to_numeric(df['Popularity'], errors='coerce')
df['Vote_Average'] = pd.to_numeric(df['Vote_Average'], errors='coerce')

# Ingeniería de datos: Agregación por Año
df['Year'] = df['Release_Date'].dt.year
df_clean = df.dropna(subset=['Year', 'Popularity', 'Vote_Average'])

yearly_stats = df_clean.groupby('Year').agg(
    Movie_Count=('Title', 'count'),
    Avg_Popularity=('Popularity', 'mean'),
    Avg_Vote=('Vote_Average', 'mean')
).reset_index()

# 2. CREACIÓN DEL BUBBLE CHART (Fondo Blanco)
fig = px.scatter(
    yearly_stats,
    x="Year",
    y="Movie_Count",
    size="Avg_Popularity",
    color="Avg_Vote",
    hover_name="Year",
    size_max=60,
    title='Tendencias del Cine: Cantidad, Popularidad y Calidad por Año',
    labels={
        'Year': 'Año de Lanzamiento',
        'Movie_Count': 'Cantidad de Películas',
        'Avg_Popularity': 'Popularidad Promedio',
        'Avg_Vote': 'Calificación Promedio'
    },
    color_continuous_scale=px.colors.sequential.Viridis
)

# Configuración estética para fondo blanco
fig.update_layout(
    template='plotly_white',
    font=dict(color="#2c3e50"),
    title_font=dict(size=22, color="black"),
    xaxis=dict(
        rangeslider=dict(visible=True, bgcolor="#f8f9fa", bordercolor="#444"),
        tickcolor='black',
        linecolor='black'
    ),
    yaxis=dict(tickcolor='black', linecolor='black'),
    height=700
)

# Añadimos un borde a las burbujas para que resalten en el fondo blanco
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))

# Mostramos el gráfico
fig.show()

# 3. EXPORTACIÓN AUTOMÁTICA
nombre_html = "bubble_chart_cine.html"
fig.write_html(nombre_html)

# Generar snippet para StackBlitz
# full_html=False nos da solo el <div>, ahorrando mucho espacio
codigo_stackblitz = fig.to_html(full_html=False, include_plotlyjs='cdn')

print("\n--- Iniciando descarga del archivo HTML ---")
files.download(nombre_html)

print("\n" + "="*50)
print("COPIA ESTE CÓDIGO PARA TU STACKBLITZ")
print("="*50 + "\n")
print(codigo_stackblitz)


--- Iniciando descarga del archivo HTML ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPIA ESTE CÓDIGO PARA TU STACKBLITZ

<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>                <div id="c1d1f83c-58a8-46fd-b048-01347b4fb2af" class="plotly-graph-div" style="height:700px; width:100%;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("c1d1f83c-58a8-46fd-b048-01347b4fb2af")) {                    Plotly.newPlot(                        "c1d1f83c-58a8-46fd-b048-01347b4fb2af",                        [{"hovertemplate":"\u003cb\u003e%{hovertext}\u003c\u002fb\u003e\u003cbr\u003e\u003cbr\u003eAño de Lanzamiento=%{x}\u003cbr\u003eCantidad de Películas=%{y}\u003cbr\u003ePopularidad Promedio=%{marker.size}\u003cbr\u003eCalificación Promedio=%{marker.color}\u003cextra\u003e\u003c\u00

In [ ]:
import pandas as pd
import plotly.express as px
from google.colab import files

# 1. CARGA Y LIMPIEZA
try:
    df = pd.read_csv('mymoviedb.csv', engine='python')
except FileNotFoundError:
    print("❌ Error: No se encontró 'mymoviedb.csv'.")
    raise

# Limpieza técnica que ya dominas
if 'Poster_Url' in df.columns:
    df = df.drop(columns=['Poster_Url'])

df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
df['Year'] = df['Release_Date'].dt.year

# 2. PROCESAMIENTO DE GÉNEROS (Flattening)
# Separamos los géneros por coma y expandimos el DataFrame
df['Genre_List'] = df['Genre'].fillna('Unknown').str.split(', ')
df_exploded = df.explode('Genre_List')

# 3. AGREGACIÓN
# Filtramos desde 1980 para mejorar la visualización
df_filtered = df_exploded[df_exploded['Year'] >= 1980]
genre_year_counts = df_filtered.groupby(['Year', 'Genre_List']).size().reset_index(name='Count')

# 4. CREACIÓN DEL GRÁFICO (Adaptado a Blanco)
fig = px.bar(
    genre_year_counts,
    x="Year",
    y="Count",
    color="Genre_List",
    title="Distribución Anual de Películas por Género (Desde 1980)",
    labels={'Count': 'Cantidad de Películas', 'Year': 'Año de Estreno', 'Genre_List': 'Género'},
    template="plotly_white", # CAMBIO: Fondo blanco
    barmode='stack'
)

# 5. AJUSTES ESTÉTICOS PRO
fig.update_layout(
    font=dict(color="#2c3e50"),
    title_font=dict(size=22, color="black"),
    xaxis=dict(
        rangeslider=dict(visible=True, bgcolor="#f8f9fa"),
        tickcolor='black'
    ),
    yaxis=dict(gridcolor='#e0e0e0'),
    hovermode="x unified", # Muestra todos los datos del año en un solo cuadro
    legend_title_text='Géneros cinematográficos:',
    height=700
)

# Mostramos el gráfico
fig.show()

# 6. EXPORTACIÓN AUTOMÁTICA
nombre_html = "distribucion_generos.html"
fig.write_html(nombre_html)

# Generar snippet para StackBlitz
codigo_stackblitz = fig.to_html(full_html=False, include_plotlyjs='cdn')

print("\n--- Iniciando descarga del archivo HTML ---")
files.download(nombre_html)

print("\n" + "="*50)
print("COPIA ESTE CÓDIGO PARA TU STACKBLITZ")
print("="*50 + "\n")
print(codigo_stackblitz)


--- Iniciando descarga del archivo HTML ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPIA ESTE CÓDIGO PARA TU STACKBLITZ

<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>                <div id="46060497-dfb2-4158-824c-fe0a41005f2f" class="plotly-graph-div" style="height:700px; width:100%;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("46060497-dfb2-4158-824c-fe0a41005f2f")) {                    Plotly.newPlot(                        "46060497-dfb2-4158-824c-fe0a41005f2f",                        [{"alignmentgroup":"True","hovertemplate":"Género=Action\u003cbr\u003eAño de Estreno=%{x}\u003cbr\u003eCantidad de Películas=%{y}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"Action","marker":{"color":"#636efa","pattern":{"shape":""}},"name":"Action","offsetgroup":"Acti

In [ ]:
import pandas as pd
import plotly.express as px
from google.colab import files

# 1. CARGA Y LIMPIEZA
try:
    df = pd.read_csv('mymoviedb.csv', engine='python')
except FileNotFoundError:
    print("❌ Error: No se encontró 'mymoviedb.csv'.")
    raise

# Limpieza: Eliminamos Poster_Url y preparamos fechas
if 'Poster_Url' in df.columns:
    df = df.drop(columns=['Poster_Url'])

df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
df['Year'] = df['Release_Date'].dt.year

# 2. PROCESAMIENTO DE GÉNEROS (Flattening)
df['Genre_List'] = df['Genre'].fillna('Unknown').str.split(', ')
df_exploded = df.explode('Genre_List')

# 3. IDENTIFICAR EL TOP 5 GÉNEROS GLOBAL
top_5_names = df_exploded['Genre_List'].value_counts().head(5).index.tolist()

# 4. FILTRAR DATASET PARA EL TOP 5 (Desde 1970)
df_top_5 = df_exploded[(df_exploded['Genre_List'].isin(top_5_names)) & (df_exploded['Year'] >= 1970)]
counts_top_5 = df_top_5.groupby(['Year', 'Genre_List']).size().reset_index(name='Movie_Count')

# 5. CREACIÓN DEL GRÁFICO DE LÍNEAS (Fondo Blanco)
fig = px.line(
    counts_top_5,
    x="Year",
    y="Movie_Count",
    color="Genre_List",
    markers=True,
    title=f"Evolución Histórica: Top 5 Géneros Dominantes",
    labels={'Movie_Count': 'Cantidad de Películas', 'Year': 'Año de Estreno', 'Genre_List': 'Género'},
    template="plotly_white" # CAMBIO: Adaptado a fondo blanco
)

# 6. CONFIGURACIÓN DINÁMICA Y ESTÉTICA
fig.update_layout(
    font=dict(color="#2c3e50"),
    title_font=dict(size=22, color="black"),
    hovermode="x unified",
    xaxis=dict(
        rangeslider=dict(visible=True, bgcolor="#f8f9fa"),
        tickcolor='black',
        linecolor='black'
    ),
    yaxis=dict(gridcolor='#e0e0e0'),
    legend_title_text='Géneros Top:',
    height=650
)

# Hacer las líneas un poco más gruesas para mejor visibilidad
fig.update_traces(line=dict(width=3), marker=dict(size=6))

# Mostrar gráfico
fig.show()

# 7. EXPORTACIÓN AUTOMÁTICA
nombre_html = "evolucion_generos_top5.html"
fig.write_html(nombre_html)

# Generar snippet para StackBlitz
codigo_stackblitz = fig.to_html(full_html=False, include_plotlyjs='cdn')

print("\n--- Descargando reporte HTML ---")
files.download(nombre_html)

print("\n" + "="*50)
print("COPIA ESTE CÓDIGO PARA TU STACKBLITZ")
print("="*50 + "\n")
print(codigo_stackblitz)


--- Descargando reporte HTML ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPIA ESTE CÓDIGO PARA TU STACKBLITZ

<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>                <div id="b983dbbc-a367-44d2-9306-d32364d9f6b5" class="plotly-graph-div" style="height:650px; width:100%;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("b983dbbc-a367-44d2-9306-d32364d9f6b5")) {                    Plotly.newPlot(                        "b983dbbc-a367-44d2-9306-d32364d9f6b5",                        [{"hovertemplate":"Género=Action\u003cbr\u003eAño de Estreno=%{x}\u003cbr\u003eCantidad de Películas=%{y}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"Action","line":{"color":"#636efa","dash":"solid","width":3},"marker":{"symbol":"circle","size":6},"mode":"lines+markers